# Stock Price Prediction with LSTM

In [5]:
import pandas as pd
import yfinance as yf
from datetime import date, timedelta

today = date.today()

end_date = today.strftime("%Y-%m-%d")
start_date = (today - timedelta(days=5000)).strftime("%Y-%m-%d")

data = yf.download(
    "AAPL",
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False
)

# Remove MultiIndex
data.columns = data.columns.get_level_values(0)

# Reset index so Date becomes a normal column
data.reset_index(inplace=True)

print(data.columns)
print(data.head())

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')
Price       Date  Adj Close      Close       High        Low       Open  \
0     2012-11-09  16.521902  19.537857  19.817142  19.061428  19.300714   
1     2012-11-12  16.394159  19.386786  19.803572  19.237499  19.791071   
2     2012-11-13  16.396271  19.389286  19.660000  19.155714  19.246786   
3     2012-11-14  16.214455  19.174286  19.551786  19.149286  19.482143   
4     2012-11-15  15.874390  18.772142  19.267857  18.665001  19.197500   

Price     Volume  
0      929913600  
1      515802000  
2      532949200  
3      477170400  
4      789910800  


In [6]:
data.tail()

Price,Date,Adj Close,Close,High,Low,Open,Volume
3435,2026-07-13,317.309998,317.309998,323.450012,315.779999,317.019989,43257800
3436,2026-07-14,314.859985,314.859985,316.190002,311.910004,313.760010,36336800
3437,2026-07-15,327.500000,327.500000,328.730011,317.320007,317.619995,60957600
3438,2026-07-16,333.260010,333.260010,334.679993,326.790009,328.010010,62970600
3439,2026-07-17,333.739990,333.739990,334.989990,329.000000,331.980011,63365300


In [7]:
import plotly.graph_objects as go
figure = go.Figure(data=[go.Candlestick(x=data["Date"],
                                        open=data["Open"], 
                                        high=data["High"],
                                        low=data["Low"], 
                                        close=data["Close"])])
figure.update_layout(title = "Apple Stock Price Analysis", 
                     xaxis_rangeslider_visible=False)
figure.show()

In [8]:
correlation = data.corr()
print(correlation["Close"].sort_values(ascending=False))

Price
Close        1.000000
Adj Close    0.999964
High         0.999882
Low          0.999881
Open         0.999736
Date         0.942163
Volume      -0.543646
Name: Close, dtype: float64


In [9]:
x = data[["Open", "High", "Low", "Volume"]]
y = data["Close"]
# convert to numpy and ensure float32 for model
x = x.to_numpy().astype('float32')
y = y.to_numpy().astype('float32')
y = y.reshape(-1, 1)
# reshape for LSTM: (samples, timesteps, features)
# treat the 4 columns as 4 timesteps with 1 feature each
x = x.reshape((x.shape[0], x.shape[1], 1))
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

In [10]:
from keras.models import Sequential
from keras.layers import Dense, LSTM
model = Sequential()
model.add(LSTM(128, return_sequences=True, input_shape= (xtrain.shape[1], 1)))
model.add(LSTM(64, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))
model.summary()

C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 4, 128)         │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 25)             │         1,625 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,619 (459.45 KB)

 Trainable params: 117,619 (459.45 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(xtrain, ytrain, batch_size=1, epochs=30)

Epoch 1/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - loss: 1166.8079
Epoch 2/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 61.8066
Epoch 3/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 49.4457
Epoch 4/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 44.0160
Epoch 5/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 38.5311
Epoch 6/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 32.2002
Epoch 7/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 34.3808
Epoch 8/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 34.3859
Epoch 9/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 21.7343
Epoch 10/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 36.1941
Epoch 11/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 24.4477
Epoch 12/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 25.8191
Epoch 13/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 35.1075
Epoch 14/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 26.2325

In [12]:
import numpy as np
# features = [Open, High, Low, Volume] - match training features
features = np.array([[177.089996, 180.419998, 177.070007, 74919600]], dtype='float32')
# reshape to (samples, timesteps, features) like training input
features = features.reshape((features.shape[0], features.shape[1], 1))
model.predict(features)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step


array([[181.45326]], dtype=float32)